# Prediccion de abandono de clientes (Churn)

Este notebook construye un modelo de clasificacion para predecir si un
cliente de una empresa de telecomunicaciones va a abandonar el servicio,
usando regresion logistica sobre datos demograficos, de contrato y de uso.

## Carga de datos

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, classification_report

model = LogisticRegression(random_state=42, max_iter=1000)
scaler = StandardScaler()
df = pd.read_csv('../data/processed/churn_clean.csv')

print(f"Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head()

Dimensiones: 7043 filas x 21 columnas


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Contexto: hallazgos clave del EDA

- Dataset limpio, sin nulos ni duplicados (TotalCharges ya convertido a numerico)
- **Contract** es el predictor mas fuerte: 42.7% de churn en mensual vs 2.8% en 2 anios
- **tenure**: clientes que se van llevan en promedio 18 meses vs 38 de los que se quedan
- **MonthlyCharges**: clientes que se van pagan en promedio $74 vs $61
- Target moderadamente desbalanceado: 73.46% No / 26.54% Yes — Recall va a ser clave

## Seleccion de features y encoding

In [23]:
#descartar customer ID porque no aporta info predictiva
df_model = df.drop(columns=['customerID'])

columnas_addon = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                   'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in columnas_addon:
    df_model[col] = df_model[col].replace({'No internet service': 'No', 'No phone service': 'No'})


#mapear target y columnas binarias yes/no a 0/1
binarias = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn'] + columnas_addon
for col in binarias:
    df_model[col] = df_model[col].map({'Yes': 1, 'No': 0})
    
#gender es binaria pero no es yes/no
df_model['gender'] = df_model['gender'].map({'Male': 1, 'Female': 0})

#one-hot encoding para el resto de columnas categoricas
columnas_multi = ['InternetService', 'Contract', 'PaymentMethod']
dummy_cols = pd.get_dummies(df_model[columnas_multi], drop_first=True).columns
df_model = pd.get_dummies(df_model, columns=columnas_multi, drop_first=True)
df_model[dummy_cols] = df_model[dummy_cols].astype(int)

X = df_model.drop(columns=['Churn'])
y = df_model['Churn']

print(f"Dimensiones — X: {X.shape} | y: {y.shape}")
print(f"\nColumnas: {X.columns.tolist()}")

Dimensiones — X: (7043, 23) | y: (7043,)

Columnas: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'InternetService_Fiber optic', 'InternetService_No', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


## Division train/test

In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  #mantiene la proporcion 73.46/26.54 en train y test
)

print(f"Entrenamiento: {X_train.shape[0]} filas")
print(f"Test: {X_test.shape[0]} filas")
print(f"\nProporcion churn en train: {y_train.mean():.4f}")
print(f"Proporcion churn en test: {y_test.mean():.4f}")

Entrenamiento: 5634 filas
Test: 1409 filas

Proporcion churn en train: 0.2654
Proporcion churn en test: 0.2654


## Escalado de features

In [25]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Media en train (debe ser ~0):", X_train_scaled.mean(axis=0).round(4)[:5])
print("Desviacion en train (debe ser ~1):", X_train_scaled.std(axis=0).round(4)[:5])

Media en train (debe ser ~0): [ 0.  0. -0.  0. -0.]
Desviacion en train (debe ser ~1): [1. 1. 1. 1. 1.]


## Entrenamiento del modelo

In [26]:

model.fit(X_train_scaled, y_train)

print("Modelo entrenado.")
print(f"Intercepto (B0): {model.intercept_[0]:.4f}")
print(f"Numero de coeficientes: {len(model.coef_[0])}")

Modelo entrenado.
Intercepto (B0): -1.7076
Numero de coeficientes: 23


## Evaluacion del modelo

In [27]:
y_pred = model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print()
print("Matriz de confusion:")
print(confusion_matrix(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))

Accuracy:  0.8070
Precision: 0.6594
Recall:    0.5642

Matriz de confusion:
[[926 109]
 [163 211]]

              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.56      0.61       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409



## Ajuste por desbalance de clases

In [28]:
model_balanced = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
model_balanced.fit(X_train_scaled, y_train)

y_pred_balanced = model_balanced.predict(X_test_scaled)

accuracy_b = accuracy_score(y_test, y_pred_balanced)
precision_b = precision_score(y_test, y_pred_balanced)
recall_b = recall_score(y_test, y_pred_balanced)

print("Modelo original vs balanceado:")
print(f"{'Metrica':<12}{'Original':<12}{'Balanceado'}")
print(f"{'Accuracy':<12}{accuracy:<12.4f}{accuracy_b:.4f}")
print(f"{'Precision':<12}{precision:<12.4f}{precision_b:.4f}")
print(f"{'Recall':<12}{recall:<12.4f}{recall_b:.4f}")
print()
print("Matriz de confusion (balanceado):")
print(confusion_matrix(y_test, y_pred_balanced))

Modelo original vs balanceado:
Metrica     Original    Balanceado
Accuracy    0.8070      0.7388
Precision   0.6594      0.5052
Recall      0.5642      0.7807

Matriz de confusion (balanceado):
[[749 286]
 [ 82 292]]
